# Notebook 03: Add a New Problem Scaffold (DCC26)

Reference implementation of a minimal, reproducible benchmark problem scaffold.


**Edit-safe start:** this notebook opens from GitHub in read-only source mode. Use **File -> Save a copy in Drive** before running edits so your changes stay in your own workspace.


## Notebook map

This notebook is written as a standalone lab chapter:
- context first,
- implementation second,
- interpretation third.

If you are following asynchronously, run cells in order and use the success checks to validate each stage before moving on.


## Standalone guide

Use this as a pattern for structuring new benchmark problems with explicit contracts and validation checks.


## What makes a new problem benchmark-ready

Benchmark value comes from clarity and comparability, not only simulator sophistication.


In [ ]:
# Colab/local dependency bootstrap
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
FORCE_INSTALL = False  # Set True to force reinstall outside Colab
PACKAGES = ['engibench[beams2d]', 'matplotlib', 'gymnasium']

if IN_COLAB or FORCE_INSTALL:
    print('Installing dependencies...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *PACKAGES])
    print('Dependency install complete.')
else:
    print('Skipping install (using current environment).')


### Step 1 - Import scaffold dependencies

Keep imports minimal and interface-focused.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Annotated

import numpy as np
from gymnasium import spaces

from engibench.constraint import bounded
from engibench.constraint import constraint
from engibench.core import ObjectiveDirection
from engibench.core import OptiStep
from engibench.core import Problem


### Step 2 - Implement battery cold-plate problem contract

Ensure methods are deterministic and constraints/objectives are semantically explicit.


In [ ]:
class BatteryColdPlate2DProblem(Problem[np.ndarray]):
    """Scaffold for a battery cold-plate topology problem (not currently in EngiBench)."""

    version = 0
    objectives = (
        ("max_temperature_c", ObjectiveDirection.MINIMIZE),
        ("flow_penalty", ObjectiveDirection.MINIMIZE),
    )

    @dataclass
    class Conditions:
        heat_load_left: Annotated[float, bounded(lower=0.1, upper=2.0)] = 1.0
        heat_load_right: Annotated[float, bounded(lower=0.1, upper=2.0)] = 1.0
        inlet_temp_c: Annotated[float, bounded(lower=10.0, upper=40.0)] = 25.0
        flow_budget: Annotated[float, bounded(lower=0.15, upper=0.65)] = 0.35

    @dataclass
    class Config(Conditions):
        resolution: Annotated[int, bounded(lower=16, upper=96)] = 32
        max_iter: Annotated[int, bounded(lower=1, upper=200)] = 30
        solver_iters: Annotated[int, bounded(lower=20, upper=500)] = 120
        min_channel_fraction: Annotated[float, bounded(lower=0.05, upper=0.60)] = 0.18
        max_channel_fraction: Annotated[float, bounded(lower=0.10, upper=0.85)] = 0.55
        max_edge_density: Annotated[float, bounded(lower=0.01, upper=1.00)] = 0.28

    dataset_id = "IDEALLab/battery_cold_plate_2d_v0"  # placeholder for future dataset integration
    container_id = None

    def __init__(self, seed: int = 0, **kwargs):
        super().__init__(seed=seed)
        self.config = self.Config(**kwargs)
        self.conditions = self.Conditions(
            heat_load_left=self.config.heat_load_left,
            heat_load_right=self.config.heat_load_right,
            inlet_temp_c=self.config.inlet_temp_c,
            flow_budget=self.config.flow_budget,
        )
        self.design_space = spaces.Box(
            low=0.0,
            high=1.0,
            shape=(self.config.resolution, self.config.resolution),
            dtype=np.float32,
        )

        @constraint
        def channel_fraction(design: np.ndarray, min_channel_fraction: float, max_channel_fraction: float, **_) -> None:
            cf = float(np.mean(1.0 - design))
            assert min_channel_fraction <= cf <= max_channel_fraction, (
                f"channel_fraction={cf:.3f} outside [{min_channel_fraction:.3f}, {max_channel_fraction:.3f}]"
            )

        @constraint
        def edge_density(design: np.ndarray, max_edge_density: float, **_) -> None:
            tv = float(np.mean(np.abs(np.diff(design, axis=0))) + np.mean(np.abs(np.diff(design, axis=1))))
            assert tv <= max_edge_density, f"edge_density={tv:.3f} exceeds {max_edge_density:.3f}"

        self.design_constraints = [channel_fraction, edge_density]

    def _heat_map(self, cfg: dict) -> np.ndarray:
        h, w = self.design_space.shape
        yy, xx = np.indices((h, w), dtype=np.float32)
        s = 0.08 * min(h, w)

        left = np.exp(-(((xx - 0.22 * w) ** 2 + (yy - 0.35 * h) ** 2) / (2.0 * s**2)))
        right = np.exp(-(((xx - 0.78 * w) ** 2 + (yy - 0.65 * h) ** 2) / (2.0 * s**2)))
        heat = cfg["heat_load_left"] * left + cfg["heat_load_right"] * right
        return heat.astype(np.float32)

    def _solve_temperature(self, conductivity: np.ndarray, heat: np.ndarray, inlet_temp: float, n_iter: int) -> np.ndarray:
        T = np.full_like(conductivity, float(inlet_temp), dtype=np.float32)
        ambient = float(inlet_temp) + 5.0

        for _ in range(int(n_iter)):
            T_old = T.copy()

            k_c = conductivity[1:-1, 1:-1]
            k_e = 0.5 * (k_c + conductivity[1:-1, 2:])
            k_w = 0.5 * (k_c + conductivity[1:-1, :-2])
            k_n = 0.5 * (k_c + conductivity[:-2, 1:-1])
            k_s = 0.5 * (k_c + conductivity[2:, 1:-1])

            numer = (
                k_e * T_old[1:-1, 2:]
                + k_w * T_old[1:-1, :-2]
                + k_n * T_old[:-2, 1:-1]
                + k_s * T_old[2:, 1:-1]
                + 0.02 * heat[1:-1, 1:-1]
            )
            denom = k_e + k_w + k_n + k_s + 1e-6
            T[1:-1, 1:-1] = numer / denom

            T[:, 0] = inlet_temp
            T[:, -1] = 0.7 * T[:, -2] + 0.3 * ambient
            T[0, :] = T[1, :]
            T[-1, :] = T[-2, :]

        return T

    def simulate(self, design: np.ndarray, config: dict | None = None) -> np.ndarray:
        cfg = {**self.__dict__["config"].__dict__, **(config or {})}

        x = np.clip(design.astype(np.float32), 0.0, 1.0)
        channel = 1.0 - x

        k_channel = 0.25
        k_solid = 4.5
        conductivity = k_channel + x * (k_solid - k_channel)

        heat = self._heat_map(cfg)
        T = self._solve_temperature(conductivity, heat, cfg["inlet_temp_c"], cfg["solver_iters"])

        max_temp = float(np.max(T))
        temp_std = float(np.std(T))

        channel_fraction = float(np.mean(channel))
        edge_density = float(np.mean(np.abs(np.diff(channel, axis=0))) + np.mean(np.abs(np.diff(channel, axis=1))))
        flow_penalty = abs(channel_fraction - cfg["flow_budget"]) + 0.15 * edge_density + 0.05 * temp_std

        return np.array([max_temp, float(flow_penalty)], dtype=np.float32)

    def optimize(self, starting_point: np.ndarray, config: dict | None = None):
        cfg = {**self.__dict__["config"].__dict__, **(config or {})}
        x = np.clip(starting_point.astype(np.float32), 0.0, 1.0)
        history = []

        h, w = x.shape
        yy, xx = np.indices((h, w), dtype=np.float32)
        thermal_bias = np.exp(-(((xx - 0.50 * w) ** 2 + (yy - 0.50 * h) ** 2) / (2.0 * (0.28 * min(h, w)) ** 2)))
        thermal_bias = (thermal_bias - thermal_bias.min()) / (thermal_bias.max() - thermal_bias.min() + 1e-8)

        target_density = np.clip(1.0 - cfg["flow_budget"], 0.2, 0.9)

        for step in range(cfg["max_iter"]):
            neighbor = (
                x
                + np.roll(x, 1, axis=0)
                + np.roll(x, -1, axis=0)
                + np.roll(x, 1, axis=1)
                + np.roll(x, -1, axis=1)
            ) / 5.0
            x = 0.70 * x + 0.20 * neighbor + 0.08 * target_density + 0.02 * thermal_bias
            x = np.clip(x, 0.0, 1.0)

            history.append(OptiStep(obj_values=self.simulate(x, cfg), step=step))

        return x, history

    def render(self, design: np.ndarray, *, open_window: bool = False):
        import matplotlib.pyplot as plt

        fig, ax = plt.subplots(figsize=(4.2, 4.2))
        im = ax.imshow(design, cmap="inferno", vmin=0, vmax=1)
        ax.set_title("BatteryColdPlate2D design (1=solid, 0=channel)")
        ax.axis("off")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        if open_window:
            plt.show()
        return fig, ax

    def random_design(self):
        x = self.np_random.random(self.design_space.shape).astype(np.float32)
        for _ in range(3):
            x = (
                x
                + np.roll(x, 1, axis=0)
                + np.roll(x, -1, axis=0)
                + np.roll(x, 1, axis=1)
                + np.roll(x, -1, axis=1)
            ) / 5.0
        return np.clip(x, 0.0, 1.0), -1


### Step 3 - Smoke-test the scaffold

Validate behavior with simple checks before scaling to real domains.


In [ ]:
problem = BatteryColdPlate2DProblem(
    seed=42,
    resolution=32,
    max_iter=20,
    heat_load_left=1.4,
    heat_load_right=1.1,
    inlet_temp_c=24.0,
    flow_budget=0.33,
)
start, _ = problem.random_design()

cfg = {
    'heat_load_left': 1.4,
    'heat_load_right': 1.1,
    'inlet_temp_c': 24.0,
    'flow_budget': 0.33,
    'resolution': 32,
    'max_iter': 20,
    'solver_iters': 100,
    'min_channel_fraction': 0.18,
    'max_channel_fraction': 0.55,
    'max_edge_density': 0.35,
}

print('design space:', problem.design_space)
print('objectives:', problem.objectives)
print('conditions:', problem.conditions)

viol = problem.check_constraints(start, config=cfg)
print('constraint violations:', len(viol))

obj0 = problem.simulate(start, config=cfg)
opt_design, history = problem.optimize(start, config=cfg)
objf = problem.simulate(opt_design, config=cfg)

print('initial objectives [max_temp_c, flow_penalty]:', obj0.tolist())
print('final objectives   [max_temp_c, flow_penalty]:', objf.tolist())
print('optimization steps:', len(history))

problem.render(opt_design)


## Mapping to real EngiBench contributions

Use this template to onboard new domains while preserving common evaluation semantics.


## Contribution checklist

Check for leakage risks, undocumented defaults, and missing reproducibility metadata before contribution.


## Troubleshooting

If a section fails, do not continue downstream. Fix locally first, then rerun the section and its immediate checks.
This notebook is intentionally staged so failures are localized.


## Takeaways

Before closing, record three points:
1. What conclusion is directly supported by your metrics?
2. What remains uncertain (and why)?
3. What extra experiment would you run next to reduce that uncertainty?
